# Delta Lake Tables
Use this notebook to explore Delta Lake functionality

In [1]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# Define the schema
schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

# df is a Spark DataFrame containing the CSV data "products.csv"
df = spark.read.format("csv").option("header", "true").schema(schema).load("Files/products/products.csv")
display(df)

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 709c9f16-aece-4a9c-9b1c-a7652e96497c)

In [3]:
df.write.format("delta").mode("overwrite").saveAsTable("managed_products")

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 5, Finished, Available, Finished)

In [ ]:
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(
      "external_products", 
      path="abfss://workspace@onelake.dfs.fabric.microsoft.com/yourLH.Lakehouse/Files/external_products"
  )


StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 6, Finished, Available, Finished)

In [5]:
%%sql
DESCRIBE FORMATTED managed_products;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 7, Finished, Available, Finished)

<Spark SQL result set with 12 rows and 3 fields>

In [7]:
%%sql
DESCRIBE FORMATTED external_products;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 9, Finished, Available, Finished)

<Spark SQL result set with 12 rows and 3 fields>

In [8]:
%%sql
DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 11, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/external_products';

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 12, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
%%sql
SELECT * FROM products;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 13, Finished, Available, Finished)

<Spark SQL result set with 295 rows and 4 fields>

In [11]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 14, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

In [15]:
%%sql
SELECT *, ROUND((ListPrice / 0.9)) AS Original_Price
FROM products
WHERE Category = 'Mountain Bikes'
LIMIT 10;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 18, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 5 fields>

In [16]:
%%sql
DESCRIBE HISTORY products;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 19, Finished, Available, Finished)

<Spark SQL result set with 2 rows and 15 fields>

In [17]:
delta_table_path = 'Files/external_products'

# Get current data
current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

# Get version 0 data
original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data)

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 20, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 351af4da-6caf-4838-80a5-744ed5935d9d)

SynapseWidget(Synapse.DataFrame, 2a661ac2-2037-4767-9db4-0e879a1cf9ac)

In [18]:
%%sql
-- Create temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 22, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [20]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 24, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 2 fields>

In [21]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(6))

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 25, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7d5a742c-f58b-4ae2-8178-766a6b8c0c1a)

## Streaming Data with PySpark

In [23]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# Create stream that reads data from the folder using JSON schema
jsonSchema = StructType([
    StructField("device", StringType(), False),
    StructField("status", StringType(), False)
]) 
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

# Write event data to the folder
device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 27, Finished, Available, Finished)

Source stream created...


In [25]:
# Write the stream to a delta table
delta_stream_table_path = 'Tables/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 29, Finished, Available, Finished)

Streaming to delta sink...


In [26]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 30, Finished, Available, Finished)

<Spark SQL result set with 9 rows and 2 fields>

In [27]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data.txt", more_data, True)

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 31, Finished, Available, Finished)

True

In [29]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 33, Finished, Available, Finished)

<Spark SQL result set with 16 rows and 2 fields>

In [30]:
# Save resources
deltastream.stop()

StatementMeta(, 31775884-534b-4789-bb4b-715a960dfc06, 34, Finished, Available, Finished)

---